# LASERNet - CNN-LSTM Microstructure Evolution Prediction

This notebook trains a CNN-LSTM model for predicting material microstructure evolution on Google Colab.

**Hardware Requirements:**
- GPU: Required (T4 or better recommended)
- RAM: 12GB+ recommended
- Disk: ~5GB for data + checkpoints

**Setup:**
1. Runtime → Change runtime type → GPU (T4, A100, or V100)
2. Upload your data to Google Drive or download it
3. Run all cells in order

## 1. Check GPU Availability

In [ ]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: No GPU detected! Training will be very slow.")
    print("Go to Runtime → Change runtime type → Hardware accelerator → GPU")

## 2. Clone Repository (or Upload Files)

In [ ]:
# Option 1: Clone from GitHub (if you have uploaded it)
# !git clone https://github.com/yourusername/LASERNet.git
# %cd LASERNet

# Option 2: Upload files manually
# Use the file upload button on the left sidebar to upload:
# - src/ folder (all .py files)
# - data/ folder (your TIFF files)

# Option 3: Mount Google Drive and access files there
from google.colab import drive
drive.mount('/content/drive')

# Navigate to your project folder in Drive
# %cd /content/drive/MyDrive/LASERNet

## 3. Install Dependencies

In [ ]:
# Install required packages
!pip install -q pillow tqdm matplotlib

# PyTorch and torchvision are pre-installed on Colab
print("✓ Dependencies installed")

## 4. Verify Data Structure

In [ ]:
import os
from pathlib import Path

# Check if data directory exists
data_dir = Path("data")
if data_dir.exists():
    samples = sorted([d.name for d in data_dir.iterdir() if d.is_dir() and d.name.isdigit()])
    print(f"✓ Found {len(samples)} samples: {samples}")
    
    # Check first sample
    if samples:
        first_sample = data_dir / samples[0]
        files = list(first_sample.glob("*.tiff"))
        print(f"\nFiles in sample {samples[0]}:")
        for f in sorted(files):
            size_mb = f.stat().st_size / 1e6
            print(f"  - {f.name} ({size_mb:.1f} MB)")
else:
    print("⚠️ Data directory not found!")
    print("Please upload your data to the 'data/' folder")

## 5. Test Configuration

In [ ]:
from src.config import Config

config = Config()
config.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Adjust settings for Colab
config.BATCH_SIZE = 4  # Adjust based on GPU memory
config.NUM_WORKERS = 2  # Colab has limited CPU cores
config.NUM_EPOCHS = 100  # Shorter for testing

config.print_config()

## 6. Test Dataset Loading

In [ ]:
from src.dataset import create_dataloaders

print("Creating dataloaders...")
train_loader, val_loader = create_dataloaders(config)

print(f"\nTrain batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")

# Test loading a batch
print("\nLoading test batch...")
for inputs, targets in train_loader:
    print(f"Input shape: {inputs.shape}")
    print(f"Target shape: {targets.shape}")
    print(f"Input range: [{inputs.min():.3f}, {inputs.max():.3f}]")
    print(f"Target range: [{targets.min():.3f}, {targets.max():.3f}]")
    break

print("\n✓ Dataset loading successful!")

## 7. Create Model

In [ ]:
from src.model import get_model
from src.utils import count_parameters, get_device

device = get_device(config.DEVICE)
print(f"Using device: {device}")

# Create model (choose 'cnn_lstm' or 'conv_lstm')
model_type = "cnn_lstm"  # or "conv_lstm" for smaller model
print(f"\nCreating {model_type} model...")

model = get_model(config, model_type)
model = model.to(device)

num_params = count_parameters(model)
print(f"Model parameters: {num_params:,}")
print(f"Model size: ~{num_params * 4 / 1e6:.1f} MB")

# Test forward pass
print("\nTesting forward pass...")
test_input = torch.randn(1, config.INPUT_CHANNELS, config.PATCH_SIZE, config.PATCH_SIZE).to(device)
with torch.no_grad():
    test_output = model(test_input)
print(f"Input: {test_input.shape} → Output: {test_output.shape}")
print("\n✓ Model creation successful!")

## 8. Training Setup

In [ ]:
import torch.optim as optim
from src.utils import set_seed, EarlyStopping

# Set random seed for reproducibility
set_seed(config.RANDOM_SEED)

# Loss function
criterion = torch.nn.MSELoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=config.LEARNING_RATE,
    weight_decay=config.WEIGHT_DECAY
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=config.LR_FACTOR,
    patience=config.LR_PATIENCE,
    min_lr=config.LR_MIN
)

# Early stopping
early_stopping = EarlyStopping(patience=20, verbose=True)

# Mixed precision training
use_amp = config.USE_MIXED_PRECISION and device.type == "cuda"
if use_amp:
    from torch.cuda.amp import autocast, GradScaler
    scaler = GradScaler()
    print("✓ Using mixed precision training")

print("✓ Training setup complete")

## 9. Training Loop

In [ ]:
from tqdm.notebook import tqdm
from src.utils import AverageMeter, calculate_metrics, visualize_sample
import matplotlib.pyplot as plt

# Create output directories
config.create_dirs()

# Training history
train_losses = []
val_losses = []
best_val_loss = float('inf')

print(f"Starting training for {config.NUM_EPOCHS} epochs...\n")

for epoch in range(1, config.NUM_EPOCHS + 1):
    # Training
    model.train()
    train_loss = AverageMeter()
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{config.NUM_EPOCHS} [Train]", leave=False)
    for inputs, targets in pbar:
        inputs = inputs.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        
        if use_amp:
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP_MAX_NORM)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRAD_CLIP_MAX_NORM)
            optimizer.step()
        
        train_loss.update(loss.item(), inputs.size(0))
        pbar.set_postfix({'loss': f'{train_loss.avg:.6f}'})
    
    train_losses.append(train_loss.avg)
    
    # Validation
    model.eval()
    val_loss = AverageMeter()
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc=f"Epoch {epoch}/{config.NUM_EPOCHS} [Val]", leave=False)
        for inputs, targets in pbar:
            inputs = inputs.to(device)
            targets = targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            val_loss.update(loss.item(), inputs.size(0))
            pbar.set_postfix({'loss': f'{val_loss.avg:.6f}'})
    
    val_losses.append(val_loss.avg)
    
    # Update learning rate
    scheduler.step(val_loss.avg)
    
    # Print progress
    print(f"Epoch {epoch:3d}/{config.NUM_EPOCHS} - "
          f"Train Loss: {train_loss.avg:.6f} - "
          f"Val Loss: {val_loss.avg:.6f} - "
          f"LR: {optimizer.param_groups[0]['lr']:.2e}")
    
    # Save best model
    if val_loss.avg < best_val_loss:
        best_val_loss = val_loss.avg
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': val_loss.avg,
        }, config.CHECKPOINT_DIR / 'best_model.pth')
        print(f"  → Saved best model (loss: {val_loss.avg:.6f})")
    
    # Early stopping
    if early_stopping(val_loss.avg):
        print("\nEarly stopping triggered!")
        break
    
    # Visualize every 10 epochs
    if epoch % 10 == 0:
        # Get a sample for visualization
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs = inputs.to(device)
                outputs = model(inputs)
                visualize_sample(
                    inputs.cpu(),
                    outputs.cpu(),
                    targets.cpu(),
                    save_path=config.OUTPUT_DIR / f"vis_epoch_{epoch:04d}.png",
                    sample_idx=0
                )
                break

print(f"\n✓ Training completed!")
print(f"Best validation loss: {best_val_loss:.6f}")

## 10. Plot Training Curves

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss', linewidth=2)
plt.plot(val_losses, label='Validation Loss', linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss', fontsize=12)
plt.title('Training and Validation Loss', fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(config.OUTPUT_DIR / 'training_curves.png', dpi=150)
plt.show()

print(f"Final train loss: {train_losses[-1]:.6f}")
print(f"Final val loss: {val_losses[-1]:.6f}")
print(f"Best val loss: {best_val_loss:.6f}")

## 11. Evaluate Model

In [ ]:
from src.utils import load_checkpoint

# Load best model
checkpoint = load_checkpoint(
    config.CHECKPOINT_DIR / 'best_model.pth',
    model,
    device=device
)

# Evaluate on validation set
model.eval()
all_metrics = []

with torch.no_grad():
    for inputs, targets in tqdm(val_loader, desc="Evaluating"):
        inputs = inputs.to(device)
        targets = targets.to(device)
        outputs = model(inputs)
        
        metrics = calculate_metrics(outputs, targets)
        all_metrics.append(metrics)

# Average metrics
avg_metrics = {}
for key in all_metrics[0].keys():
    avg_metrics[key] = sum(m[key] for m in all_metrics) / len(all_metrics)

print("\nValidation Metrics:")
print("=" * 60)
for key, value in avg_metrics.items():
    print(f"{key:20s}: {value:.6f}")
print("=" * 60)

## 12. Visualize Predictions

In [ ]:
# Get some samples for visualization
model.eval()
with torch.no_grad():
    for i, (inputs, targets) in enumerate(val_loader):
        if i >= 3:  # Show 3 samples
            break
        
        inputs = inputs.to(device)
        outputs = model(inputs)
        
        # Visualize first sample in batch
        visualize_sample(
            inputs.cpu(),
            outputs.cpu(),
            targets.cpu(),
            save_path=config.OUTPUT_DIR / f"prediction_sample_{i}.png",
            sample_idx=0
        )
        
        # Display in notebook
        from IPython.display import Image as IPImage, display
        display(IPImage(filename=str(config.OUTPUT_DIR / f"prediction_sample_{i}.png")))

## 13. Download Results

In [ ]:
# Zip checkpoint and outputs for download
!zip -r lasernet_results.zip checkpoints/ outputs/

# Download via Colab
from google.colab import files
files.download('lasernet_results.zip')

print("✓ Results ready for download!")

## 14. Save to Google Drive (Optional)

In [ ]:
# Copy results to Google Drive
!cp -r checkpoints/ /content/drive/MyDrive/LASERNet_results/checkpoints/
!cp -r outputs/ /content/drive/MyDrive/LASERNet_results/outputs/

print("✓ Results saved to Google Drive!")

## Summary

**Next Steps:**
1. Download your trained model and results
2. Use the model for inference on new samples
3. Fine-tune hyperparameters for better performance
4. Try the ConvLSTM model variant

**Tips for Better Results:**
- Train for more epochs (200-300)
- Use a larger GPU (A100 if available)
- Adjust batch size based on GPU memory
- Experiment with different learning rates
- Monitor overfitting via validation loss

**Troubleshooting:**
- Out of memory? Reduce `BATCH_SIZE` or `PATCH_SIZE`
- Slow training? Ensure GPU is enabled
- Poor results? Train longer or adjust hyperparameters